In [ ]:
# [목적] CSV 데이터와 AI 모델을 연결해 표 데이터를 질문할 준비를 합니다.
# pandas는 표를 다루고, dotenv는 API 키를 읽으며, logging은 실행 기록을 남깁니다.
import pprint
from typing import Any, Dict

import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

In [ ]:
# [목적] 표에 관한 자연어 질문을 처리할 AI 모델을 만듭니다.
# temperature=0은 같은 질문에서 결과가 크게 달라지지 않게 합니다.
model = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

In [ ]:
# [목적] AI와 데이터 파서의 결과를 보기 쉬운 형태로 출력하는 함수를 만듭니다.
# 표 데이터는 그대로 출력하면 복잡하므로 to_dict()로 일반 딕셔너리로 바꿉니다.
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)    

In [ ]:
# [목적] Titanic CSV 파일을 pandas 표(DataFrame)로 불러옵니다.
# df.head()는 데이터가 제대로 읽혔는지 첫 5행을 확인할 때 사용합니다.
df = pd.read_csv("./data/titanic.csv")
df.head()

In [ ]:
# [목적] AI가 df 표를 조회·계산할 수 있도록 DataFrame 전용 파서를 만듭니다.
# get_format_instructions()는 AI에게 표를 다루는 방법을 알려주는 안내문을 만듭니다.
parser = PandasDataFrameOutputParser(dataframe=df)

print(parser.get_format_instructions)

In [ ]:
# [목적] 표에 대한 질문을 프롬프트·AI·파서로 실행하는 전체 흐름을 만듭니다.
# chain은 질문 → AI 해석 → pandas 결과 변환 순서로 연결되고, 결과는 보기 좋게 출력합니다.
df_query = "Age column을 조회해 주세요."

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

chain = prompt | model | parser

parser_output = chain.invoke({"question": df_query})

format_parser_output(parser_output)

In [ ]:
# [목적] 이미 만든 chain으로 표의 첫 번째 행을 자연어로 조회합니다.
# 질문만 바꾸면 같은 코드로 여러 종류의 표 조회를 할 수 있습니다.
df_query = "Retrieve the first row."
parser_output = chain.invoke({"question": df_query})
format_parser_output(parser_output)

In [ ]:
# [목적] AI를 거치지 않고 pandas로 앞 5명의 나이 평균을 직접 계산합니다.
# AI가 낸 평균 결과와 비교해 계산이 맞는지 확인할 때 사용합니다.
df["Age"].head().mean()

In [23]:
# [목적] 앞 5행의 Age 평균을 AI에게 자연어로 요청합니다.
# np.float64(31.2)는 31.2와 같은 값이며 NumPy의 숫자 표시 방식입니다.
df_query = "Retrieve the average of the Ages from row 0 to 4."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(31.2)}


In [24]:
# [목적] Fare 열의 평균 요금을 AI에게 자연어로 요청합니다.
# Age 평균 예제와 같은 흐름으로 다른 열도 계산할 수 있음을 보여줍니다.
df_query = "Calculate average 'Fare' rate."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(22.19937)}


In [ ]:
# [목적] pandas로 Fare 열 평균을 직접 계산해 AI 결과와 비교합니다.
# AI 결과를 사용할 때는 이런 방식으로 중요한 계산을 검증할 수 있습니다.
df["Fare"].mean()